# Stage 2: Intent-Based Manipulation Detection (Simplified)

Notebook này sử dụng intent từ stage 1 kết hợp với LLM để phát hiện manipulation trong các cuộc đối thoại pháp lý.

## Phương pháp đơn giản hóa:
1. **Load model**: Fine-tuned Mistral-7B từ stage 1  
2. **Load data**: Dialogue + Intent từ stage 1
3. **Single LLM call**: Kết hợp dialogue và intent để detect manipulation
4. **Output**: Manipulation present, manipulator, techniques

## Model:
- Base: Mistral-7B-Instruct-v0.3 (4-bit quantized)
- Fine-tuned: Custom adapter từ stage 1

## 1. Cài đặt thư viện

Cài đặt các thư viện cần thiết cho multi-agent system và model inference.

In [1]:
!pip install -U transformers bitsandbytes pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 875.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 9.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Import thư viện

In [2]:
import json 
import pandas as pd 
import torch 
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

2025-11-18 07:09:24.219008: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763449764.671625      39 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763449764.779104      39 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 3. Cấu hình HuggingFace Token

Lấy HF token từ Kaggle secrets để tải model từ HuggingFace Hub.

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

## 4. Kiểm tra môi trường

Kiểm tra CUDA, GPU memory và PyTorch version trước khi load model.

In [4]:
# Kiểm tra môi trường CUDA
print("=== Environment Check ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")
print("=" * 50)

=== Environment Check ===
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 15.83 GB
PyTorch Version: 2.6.0+cu124


## 5. Load Base Model

Tải Mistral-7B-Instruct với cấu hình:
- **4-bit quantization** (NF4): Giảm memory footprint
- **BitsAndBytes**: Quantization backend
- **Device map auto**: Tự động phân bổ model lên GPU/CPU

In [5]:
torch.cuda.empty_cache() 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer from {base_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

print(f"Loading base model from {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)
print("Base model loaded in 4-bit")

Loading tokenizer from mistralai/Mistral-7B-Instruct-v0.3...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded in 4-bit


## 6. Load Fine-tuned Adapter & Merge

Load PEFT adapter đã được fine-tune từ stage 1:
- Kiểm tra path tồn tại trước khi load
- **Merge model** để tạo standalone model
- Set model về `eval()` mode cho inference

⚠️ **Lưu ý**: Merge sẽ cần nhiều GPU memory hơn. Nếu bị OOM, bỏ dòng merge.

In [6]:
import os

adapter_model_id = "/kaggle/input/final-model2/final_model"  # add finetuned model

# Kiểm tra path tồn tại
if not os.path.exists(adapter_model_id):
    raise FileNotFoundError(f"Adapter path not found: {adapter_model_id}")

print(f"Found adapter at: {adapter_model_id}")

try:
    model = PeftModel.from_pretrained(base_model, adapter_model_id)
    print("PEFT adapter loaded successfully")
    
except Exception as e:
    print(f"Failed to load/merge PEFT adapter: {e}")
    raise

Found adapter at: /kaggle/input/final-model2/final_model
PEFT adapter loaded successfully


## 7. Load Test Data với Intent

Load dataset chứa:
- **Dialogue**: Cuộc hội thoại giữa plaintiff và defendant
- **PLAINTIFF intent**: Ý định của nguyên đơn từ stage 1
- **DEFENDANT intent**: Ý định của bị đơn từ stage 1

In [7]:
def load_csv(file_path):
    return pd.read_csv(file_path)

csv_df = load_csv("/kaggle/input/intent/test_intent.csv")
dialogues = csv_df["Dialogue"].tolist()
plaintiff_intents = csv_df["PLAINTIFF"].tolist() 
defendant_intents = csv_df["DEFENDANT"].tolist()

## 8. Định nghĩa Manipulation Tactics

Danh sách 11 kỹ thuật manipulation được model nhận diện

In [8]:
allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion", 
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

## 9. Single LLM Function

Hàm detection đơn giản sử dụng dialogue + intent để phát hiện manipulation:
- **Input**: Dialogue + Plaintiff Intent + Defendant Intent
- **Output**: Manipulation detection + Manipulator + Techniques
- **Single call**: Không cần multi-agent, chỉ 1 lần gọi LLM

In [9]:
def detect_manipulation_with_intent(dialogue, plaintiff_intent, defendant_intent):
    """
    Single LLM call để detect manipulation sử dụng dialogue và intent
    """
    prompt = f"""<s>[INST] You are an expert in detecting manipulation in legal conversations. 
Analyze the dialogue along with the intents of both parties to determine if manipulation is present.

Dialogue: {dialogue}

Plaintiff's Intent: {plaintiff_intent}
Defendant's Intent: {defendant_intent}

Available manipulation techniques:
{', '.join(allowed_tactics)}

Instructions:
1. Determine if manipulation is present (Yes/No)
2. If yes, identify who is the primary manipulator (Plaintiff/Defendant)  
3. If yes, identify which techniques from the list are being used

Format your response EXACTLY like this:
Manipulation Present: [Yes/No]
Primary Manipulator: [Plaintiff/Defendant/None]
Techniques Used: [list techniques separated by commas, or None]
[/INST]"""

    response = generate_response(prompt, max_new_tokens=200, temperature=0.3)
    
    # Extract response after [/INST]
    model_response = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    if model_response:
        extracted_response = model_response.group(1).strip()
    else:
        extracted_response = response.strip()
    
    # Parse the response
    manipulation_present = "No"
    primary_manipulator = "None"
    techniques_used = []
    
    for line in extracted_response.split('\n'):
        line = line.strip()
        if line.lower().startswith("manipulation present:"):
            value = line.split(':', 1)[1].strip()
            if value.lower() in ['yes', 'y', '1', 'true']:
                manipulation_present = "Yes"
                
        elif line.lower().startswith("primary manipulator:"):
            value = line.split(':', 1)[1].strip()
            if value.lower() in ['plaintiff', 'p']:
                primary_manipulator = "Plaintiff"
            elif value.lower() in ['defendant', 'd']:
                primary_manipulator = "Defendant"
                
        elif line.lower().startswith("techniques used:"):
            techniques_part = line.split(':', 1)[1].strip()
            if techniques_part.lower() != "none":
                for technique in techniques_part.split(','):
                    technique = technique.strip()
                    for allowed_tactic in allowed_tactics:
                        if technique.lower() == allowed_tactic.lower():
                            techniques_used.append(allowed_tactic)
                            break
    
    return {
        "manipulation_present": manipulation_present,
        "primary_manipulator": primary_manipulator,
        "techniques": techniques_used
    }

def generate_response(prompt, max_new_tokens=1000, temperature=0.4):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

## 10. Processing & Results

Hàm đơn giản để xử lý tất cả dialogues:
- **process_all_dialogues()**: Chạy detection cho toàn bộ dataset
- **save_results_to_csv()**: Lưu kết quả ra file CSV
- **No complex workflow**: Chỉ loop qua từng dialogue và call LLM

In [10]:
def process_all_dialogues():
    """
    Xử lý tất cả dialogues trong dataset sử dụng intent và LLM
    """
    results = []
    
    for idx in range(len(dialogues)):
        print(f"\n=== Processing Dialogue {idx+1}/{len(dialogues)} ===")
        
        dialogue = dialogues[idx]
        plaintiff_intent = plaintiff_intents[idx]
        defendant_intent = defendant_intents[idx]
        
        print(f"Dialogue: {dialogue[:100]}...")
        print(f"Plaintiff Intent: {plaintiff_intent}")
        print(f"Defendant Intent: {defendant_intent}")
        
        # Single LLM call để detect manipulation
        result = detect_manipulation_with_intent(dialogue, plaintiff_intent, defendant_intent)
        
        print(f"Result:")
        print(f"- Manipulation Present: {result['manipulation_present']}")
        print(f"- Primary Manipulator: {result['primary_manipulator']}")  
        print(f"- Techniques Used: {', '.join(result['techniques']) if result['techniques'] else 'None'}")
        
        results.append(result)
    
    return results

def save_results_to_csv():
    """
    Chạy detection và lưu kết quả ra CSV
    """
    print("Starting manipulation detection using intent + LLM...")
    
    # Process all dialogues
    results = process_all_dialogues()
    
    # Prepare data for CSV
    manipulation_results = [r['manipulation_present'] for r in results]
    manipulator_results = [r['primary_manipulator'] for r in results] 
    techniques_results = [', '.join(r['techniques']) if r['techniques'] else 'None' for r in results]
    
    # Add results to dataframe
    csv_df['Manipulation_Present'] = manipulation_results
    csv_df['Primary_Manipulator'] = manipulator_results
    csv_df['Techniques_Used'] = techniques_results
    
    # Save to CSV
    output_path = "/kaggle/working/manipulation_results_simple.csv"
    csv_df.to_csv(output_path, index=False)
    
    print(f"\n=== Summary ===")
    print(f"Total dialogues processed: {len(results)}")
    print(f"Manipulation detected: {sum(1 for r in results if r['manipulation_present'] == 'Yes')}")
    print(f"No manipulation: {sum(1 for r in results if r['manipulation_present'] == 'No')}")
    print(f"Results saved to: {output_path}")
    
    return output_path

## 11. Run Simple Pipeline

In [11]:
if __name__ == "__main__":
    save_results_to_csv()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Starting manipulation detection using intent + LLM...

=== Processing Dialogue 1/155 ===
Dialogue: Judge
I understand, but my hypothetical assumes that you know all that, your clients know all that,...
Plaintiff Intent: The Plaintiff's intent, as represented by their lawyer, appears to be to argue for the application of the legal principle of proximate cause in their defense. The Plaintiff's argument is based on the precedent set in the Hemi Group and St. Paul Railway cases, which state that if there is a sufficient and independent cause operating between the wrong and the injury, it breaks the chain of causation, even if the independent cause is eminently foreseeable.
Defendant Intent: The Defendant's lawyer is attempting to argue that the Defendant did not have proximate cause for the alleged wrongdoing, even if it was foreseeable that their actions could lead to certain consequences. They are using legal precedents, specifically the St. Paul Railway case and the Hemi Group case, to 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Result:
- Manipulation Present: No
- Primary Manipulator: None
- Techniques Used: None

=== Processing Dialogue 2/155 ===
Dialogue: Judge:
Well, but if -- why -- your voluntary dismissal was without prejudice, right? Why do you have...
Plaintiff Intent: The Plaintiff's intent in this dialogue is to reopen their previously dismissed case to avoid the statute of limitations that would bar a new ADEA (Age Discrimination in Employment Act) challenge or a new application under the Federal Arbitration Act (FAA). The Plaintiff's legal team is arguing that the district court's decision to reopen the case under Rule 60(b)(6) of the Federal Rules of Civil Procedure was justified due to the unique circumstances of the case, and that this issue was not addressed below and could be considered on remand. The Plaintiff seeks to establish that the district court had jurisdiction to consider the 60(b) motion and to vacate an arbitrable award, as the Tenth Circuit Court of Appeals remanded the case back

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

Result:
- Manipulation Present: Yes
- Primary Manipulator: Plaintiff
- Techniques Used: persuasion, emotional appeal, framing the narrative

=== Processing Dialogue 153/155 ===
Dialogue: Judge: Michael Lockett, yes, you are suing Aisha Lockett, no relation, no, spelled differently, yes....
Plaintiff Intent: The Plaintiff, Michael Lockett, is seeking compensation of $5,000 from Aisha Lockett and Family Ties Homes Incorporated for consulting fees related to the establishment of a homeless shelter. His intent is to recover the costs he incurred in providing professional services, as he started working on the project out of a personal desire to help disadvantaged children, having previously worked with them in child care. He positions himself as an expert in starting businesses and nonprofit organizations, and he claims that Aisha Lockett approached him for assistance in preparing documents for the shelter.
Defendant Intent: The Defendant's intent, as revealed in the dialogue, appears to b

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Result:
- Manipulation Present: Yes
- Primary Manipulator: Plaintiff
- Techniques Used: persuasion, emotional appeal, framing the narrative, dismissal, guilt tripping

=== Processing Dialogue 154/155 ===
Dialogue: Defendant's Lawyer: We don't disagree that marriage is an important right and that she has liberty i...
Plaintiff Intent: The Plaintiff's intent, as evidenced by their statements, is to argue that there is no constitutionally protected interest for their spouse's admission to the United States, while simultaneously maintaining that if such an interest exists, the remedies provided by Congress are sufficient. The Plaintiff is attempting to defend the doctrine of consular nonreviewability, which they believe is rooted in the political branch's plenary power to determine immigration policies. They are pivoting their argument in response to the judge's questioning, acknowledging the importance of marriage and liberty interests, but arguing that these interests are not protected b

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Result:
- Manipulation Present: Yes
- Primary Manipulator: Defendant
- Techniques Used: persuasion, playing the victim, gaslighting, deflection, minimization, dismissal, guilt tripping, emotional appeal, framing the narrative, character attack

=== Processing Dialogue 155/155 ===
Dialogue: Plaintiff: I don't think he is budgeting right. I think he's spending all his money on his new fianc...
Plaintiff Intent: The Plaintiff's intent, as revealed in the dialogue, appears to be twofold. Firstly, the Plaintiff expresses frustration and anger towards the Defendant for allegedly mismanaging finances, specifically by spending a significant amount of money on his new fiancée. This suggests a desire for financial accountability and fair distribution of resources, possibly due to a shared financial agreement or obligation.
Defendant Intent: The Defendant's intent, as revealed in the dialogue, appears to be twofold. Firstly, the Defendant is attempting to justify their financial actions by redire